In [1]:
import pandas as pd

columns = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points"
]

columns += [f"Wilderness_Area{i}" for i in range(1, 5)]
columns += [f"Soil_Type{i}" for i in range(1, 41)]
columns += ["Cover_Type"]

df = pd.read_csv("covertype/covtype.data.gz", header=None)
df.columns = columns

print(df.shape)
print(df["Cover_Type"].value_counts().sort_index())

(581012, 55)
Cover_Type
1    211840
2    283301
3     35754
4      2747
5      9493
6     17367
7     20510
Name: count, dtype: int64


In [ ]:
#split
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,   # 30% temp
    stratify=df["Cover_Type"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=2/3,   # 20% test, 10% val
    stratify=temp_df["Cover_Type"],
    random_state=42
)

print(train_df.shape, val_df.shape, test_df.shape)

train_small = (
    train_df.groupby("Cover_Type", group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 4000), random_state=42))
)
val_small = val_df.sample(n=5000, random_state=42)
test_small = test_df.sample(n=5000, random_state=42)

(406708, 55) (58101, 55) (116203, 55)


In [ ]:
numeric_cols = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points"
]

wilderness_cols = [f"Wilderness_Area{i}" for i in range(1, 5)]
soil_cols = [f"Soil_Type{i}" for i in range(1, 41)]

numeric_cols = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points"
]

wilderness_cols = [f"Wilderness_Area{i}" for i in range(1, 5)]
soil_cols = [f"Soil_Type{i}" for i in range(1, 41)]

def get_active_index(row, cols):
    for i, col in enumerate(cols, start=1):
        if row[col] == 1:
            return i
    return 0

def row_to_prompt(row):
    wilderness = get_active_index(row, wilderness_cols)
    soil = get_active_index(row, soil_cols)

    parts = [
    "Forest data:",
    f"Elevation: {row['Elevation']}",
    f"Aspect: {row['Aspect']}",
    f"Slope: {row['Slope']}",
    f"Horizontal Distance to Hydrology: {row['Horizontal_Distance_To_Hydrology']}",
    f"Vertical Distance to Hydrology: {row['Vertical_Distance_To_Hydrology']}",
    f"Horizontal Distance to Roadways: {row['Horizontal_Distance_To_Roadways']}",
    f"Hillshade 9am: {row['Hillshade_9am']}",
    f"Hillshade Noon: {row['Hillshade_Noon']}",
    f"Hillshade 3pm: {row['Hillshade_3pm']}",
    f"Horizontal Distance to Fire Points: {row['Horizontal_Distance_To_Fire_Points']}",
    f"Wilderness Area: {wilderness}",
    f"Soil Type: {soil}",
    "",
    "Predict the forest cover type (1-7).",
    "Answer:"
]
    return " ; ".join(parts)

def label_to_token(label):
    return f"<C{int(label)}>"

sample_row = train_df.iloc[0]
print(row_to_prompt(sample_row))
print(label_to_token(sample_row["Cover_Type"]))

Elevation: 2823 ; Aspect: 30 ; Slope: 29 ; Horizontal_Distance_To_Hydrology: 162 ; Vertical_Distance_To_Hydrology: 46 ; Horizontal_Distance_To_Roadways: 1150 ; Hillshade_9am: 198 ; Hillshade_Noon: 165 ; Hillshade_3pm: 87 ; Horizontal_Distance_To_Fire_Points: 1746 ; Wilderness_Area: 3 ; Soil_Type: 24 ; Cover type:
<C2>


In [4]:
#building dataset
import torch
from torch.utils.data import Dataset
class CovertypeLLMDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        prompt = row_to_prompt(row)
        label_text = " " + label_to_token(row["Cover_Type"])

        full_text = prompt + label_text

        full_enc = self.tokenizer(
            full_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        prompt_enc = self.tokenizer(
            prompt,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        input_ids = full_enc["input_ids"].squeeze(0)
        attention_mask = full_enc["attention_mask"].squeeze(0)

        labels = input_ids.clone()

        prompt_len = int(prompt_enc["attention_mask"].sum().item())

        labels[:prompt_len] = -100
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [5]:
#loading decoder only 
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

special_tokens = {
    "additional_special_tokens": [f"<C{i}>" for i in range(1, 8)]
}
tokenizer.add_special_tokens(special_tokens)

model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [6]:
from torch.utils.data import Dataset

class CovertypeLLMDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        print("INIT START")
        self.df = dataframe.reset_index(drop=True)
        print("after reset_index")
        self.tokenizer = tokenizer
        print("after tokenizer assign")
        self.max_length = max_length
        print("INIT END")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        prompt = row_to_prompt(row)
        label_text = " " + label_to_token(row["Cover_Type"])
        full_text = prompt + label_text

        full_enc = self.tokenizer(
            full_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        prompt_enc = self.tokenizer(
            prompt,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        input_ids = full_enc["input_ids"].squeeze(0)
        attention_mask = full_enc["attention_mask"].squeeze(0)

        labels = input_ids.clone()
        prompt_len = int(prompt_enc["attention_mask"].sum().item())

        labels[:prompt_len] = -100
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }
    
    train_small = train_df.sample(n=10, random_state=42)
dummy_tokenizer = None

test_ds = CovertypeLLMDataset(train_small, dummy_tokenizer, max_length=128)
print("created")

INIT START
after reset_index
after tokenizer assign
INIT END
created


In [7]:
#train/val/test

train_dataset = CovertypeLLMDataset(train_small, tokenizer, max_length=256)
val_dataset = CovertypeLLMDataset(val_small, tokenizer, max_length=256)
test_dataset = CovertypeLLMDataset(test_small, tokenizer, max_length=256)

print("train size:", len(train_dataset))
print("val size:", len(val_dataset))

INIT START
after reset_index
after tokenizer assign
INIT END
INIT START
after reset_index
after tokenizer assign
INIT END
INIT START
after reset_index
after tokenizer assign
INIT END
train size: 30000
val size: 5000


In [ ]:
#training 
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score
import numpy as np

training_args = TrainingArguments(
    output_dir="./llm_covertype",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

/Users/kareena/Documents/CAP 5610/Project/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,0.398047,0.393461


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=3750, training_loss=0.7031058369954427, metrics={'train_runtime': 15219.7933, 'train_samples_per_second': 1.971, 'train_steps_per_second': 0.246, 'total_flos': 1959725629440000.0, 'train_loss': 0.7031058369954427, 'epoch': 1.0})

In [ ]:
#prediction 
def predict_label(row, model, tokenizer, max_new_tokens=5, device="cpu"):
    prompt = row_to_prompt(row)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=False)

    for i in range(1, 8):
        if f"<C{i}>" in decoded:
            return i

    return None

In [ ]:
#eval 
from sklearn.metrics import classification_report, accuracy_score
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

y_true = []
y_pred = []

sample_test_df = test_df.sample(n=2000, random_state=42)  # smaller first for speed

for _, row in sample_test_df.iterrows():
    pred = predict_label(row, model, tokenizer, device=device)

     if pred is None:
        pred = 1  # default to class 1 if no prediction
    
    y_true.append(int(row["Cover_Type"]))
    y_pred.append(pred)

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, digits=4))

Accuracy: 0.654
              precision    recall  f1-score   support

           1     0.5679    0.8408    0.6779       691
           2     0.7833    0.6155    0.6893      1022
           3     0.5632    0.9159    0.6975       107
           4     0.0000    0.0000    0.0000         9
           5     0.0000    0.0000    0.0000        39
           6     0.0000    0.0000    0.0000        57
           7     0.0000    0.0000    0.0000        75

    accuracy                         0.6540      2000
   macro avg     0.2735    0.3389    0.2950      2000
weighted avg     0.6266    0.6540    0.6238      2000



/Users/kareena/Documents/CAP 5610/Project/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/kareena/Documents/CAP 5610/Project/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/kareena/Documents/CAP 5610/Project/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

In [11]:
import sys
print(sys.executable)

import transformers
print(transformers.__version__)

/Users/kareena/Documents/CAP 5610/Project/venv/bin/python
5.6.0
